# Explanation Stability Stress Test

This notebook is Milestone 09 for the revised fraud-detection paper.

Research purpose:

- Test whether model explanations are stable across random seeds.
- Test whether feature importance rankings change when random stratified evaluation is replaced by chronological evaluation.
- Aggregate one-hot/scaled transformed features back to original governed feature names.
- Report Top-K Jaccard stability, weighted importance overlap, rank correlation, and an Explanation Instability Index.

This is an explanation-stability experiment. It does not add leakage-risk features and does not claim that native tree importances are causal explanations.

In [ ]:
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn scipy
!pip -q install -q "pyarrow>=19,<24"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive screenshot showed: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'explanation_stability_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '09_explanation_stability' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
IMPORTANCE_DIR = EXPERIMENT_ROOT / 'importance_checkpoints'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, IMPORTANCE_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 101, 202, 303, 404]
MODEL_NAMES = ['xgboost', 'lightgbm']
SPLIT_POLICIES = ['RANDOM_STRATIFIED_SAFE', 'CHRONOLOGICAL_SAFE']
TOP_K_VALUES = [3, 5, 10]

TEST_SIZE = 0.20

CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': None,
    'D3_PaySim': 1_000_000,
}

SMOKE_TEST = False
SMOKE_CAP = 50_000

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)
print('Seeds:', SEEDS)

In [ ]:
import itertools
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def metric_checkpoint_path(seed, dataset, split_policy, model_name):
    return CHECKPOINT_DIR / f'seed{seed}__{dataset}__{split_policy}__{model_name}.csv'

def importance_checkpoint_path(seed, dataset, split_policy, model_name):
    return IMPORTANCE_DIR / f'importance__seed{seed}__{dataset}__{split_policy}__{model_name}.csv'

def is_done(seed, dataset, split_policy, model_name):
    return metric_checkpoint_path(seed, dataset, split_policy, model_name).exists() and importance_checkpoint_path(seed, dataset, split_policy, model_name).exists()

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        if directory.exists():
            candidates.extend(directory.glob(f'**/{file_name}'))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name}')

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def temporal_subsample(df, order_col, cap):
    if SMOKE_TEST:
        cap = min(SMOKE_CAP, len(df))
    if cap is None or len(df) <= cap:
        return df.reset_index(drop=True)
    positions = np.linspace(0, len(df) - 1, num=cap, dtype=np.int64)
    return df.iloc[positions].reset_index(drop=True)

def load_dataset(dataset_name):
    path = DATASET_FILES[dataset_name]
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'
        df['__order__'] = pd.to_numeric(df['Time'], errors='coerce')

    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
        df['__order__'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')

    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)
        df['__order__'] = pd.to_numeric(df['step'], errors='coerce')

    else:
        raise ValueError(dataset_name)

    df = df.dropna(subset=[label_col]).sort_values('__order__').reset_index(drop=True)
    df = temporal_subsample(df, '__order__', CAPS[dataset_name])
    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    append_log(f'{dataset_name}: rows={len(df):,}, fraud={fraud:,}, rate={fraud/len(df)*100:.4f}%')
    return df, label_col

def prepare_features(df, dataset_name):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        y = df['Class'].astype(int)
        X = df[[f'V{i}' for i in range(1, 29)] + ['Amount']].copy()
        return X, y

    if dataset_name == 'D2_fraudTest':
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25
        keep = ['category', 'amt', 'state', 'city_pop', 'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years']
        return df[keep].copy(), df['is_fraud'].astype(int)

    if dataset_name == 'D3_PaySim':
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        return df[keep].copy(), df['isFraud'].astype(int)

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])
    return preprocessor, numeric_cols, categorical_cols

In [ ]:
def build_model(model_name, seed, scale_pos_weight=1.0):
    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(model_name)

def get_scores(model, X):
    return model.predict_proba(X)[:, 1]

def split_indices(y, split_policy, seed):
    y = pd.Series(y).reset_index(drop=True)
    idx = np.arange(len(y))
    if split_policy == 'RANDOM_STRATIFIED_SAFE':
        train_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y, random_state=seed)
        return np.array(train_idx), np.array(test_idx)
    if split_policy == 'CHRONOLOGICAL_SAFE':
        train_end = int(len(y) * (1.0 - TEST_SIZE))
        return idx[:train_end], idx[train_end:]
    raise ValueError(split_policy)

def evaluate_model(y_true, scores):
    pred = (scores >= 0.5).astype(int)
    return {
        'n_test': int(len(y_true)),
        'test_fraud': int(np.sum(y_true == 1)),
        'test_prevalence': float(np.mean(y_true == 1)),
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'mcc_at_050': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
    }

def map_transformed_feature_name(name, numeric_cols, categorical_cols):
    raw = name
    if raw.startswith('num__'):
        raw = raw[len('num__'):]
    elif raw.startswith('cat__'):
        raw = raw[len('cat__'):]
        for col in sorted(categorical_cols, key=len, reverse=True):
            if raw == col or raw.startswith(col + '_'):
                return col
    return raw

def aggregate_importance(preprocessor, model, numeric_cols, categorical_cols):
    transformed_names = preprocessor.get_feature_names_out()
    importances = np.asarray(model.feature_importances_, dtype=float)
    if len(importances) != len(transformed_names):
        raise RuntimeError(f'Importance length mismatch: {len(importances)} vs {len(transformed_names)}')
    rows = []
    for transformed, importance in zip(transformed_names, importances):
        raw_feature = map_transformed_feature_name(str(transformed), numeric_cols, categorical_cols)
        rows.append({'transformed_feature': str(transformed), 'raw_feature': raw_feature, 'importance': float(importance)})
    detail = pd.DataFrame(rows)
    grouped = detail.groupby('raw_feature', as_index=False)['importance'].sum()
    total = grouped['importance'].sum()
    grouped['importance_normalized'] = grouped['importance'] / total if total > 0 else 0.0
    grouped = grouped.sort_values(['importance_normalized', 'raw_feature'], ascending=[False, True]).reset_index(drop=True)
    grouped['rank'] = np.arange(1, len(grouped) + 1)
    return grouped, detail

In [ ]:
def run_one(seed, dataset_name, split_policy, model_name):
    if is_done(seed, dataset_name, split_policy, model_name):
        append_log(f'SKIP seed={seed} {dataset_name} / {split_policy} / {model_name}')
        return (
            pd.read_csv(metric_checkpoint_path(seed, dataset_name, split_policy, model_name)),
            pd.read_csv(importance_checkpoint_path(seed, dataset_name, split_policy, model_name)),
        )

    df, label_col = load_dataset(dataset_name)
    X, y = prepare_features(df, dataset_name)
    train_idx, test_idx = split_indices(y, split_policy, seed)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    if int(np.sum(y_train == 1)) == 0 or int(np.sum(y_test == 1)) == 0:
        raise RuntimeError(f'Zero-fraud partition for {dataset_name} {split_policy} seed={seed}')

    preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    scores = get_scores(model, X_test_t)
    metrics = evaluate_model(y_test.reset_index(drop=True), scores)
    importance, _detail = aggregate_importance(preprocessor, model, numeric_cols, categorical_cols)

    common = {
        'run_name': RUN_NAME,
        'timestamp': now(),
        'seed': seed,
        'dataset': dataset_name,
        'split_policy': split_policy,
        'model': model_name,
        'feature_policy': 'strict',
        'smoke_test': SMOKE_TEST,
        'rows_used': int(len(df)),
        'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
        'fit_seconds': float(fit_seconds),
        'train_n': int(len(y_train)),
        'train_fraud': int(np.sum(y_train == 1)),
    }
    metric_row = pd.DataFrame([{**common, **metrics}])
    for key, value in common.items():
        importance[key] = value
    importance = importance[list(common.keys()) + ['raw_feature', 'importance', 'importance_normalized', 'rank']]

    write_csv_atomic(metric_checkpoint_path(seed, dataset_name, split_policy, model_name), metric_row)
    write_csv_atomic(importance_checkpoint_path(seed, dataset_name, split_policy, model_name), importance)
    top = ', '.join(importance.head(3)['raw_feature'].tolist())
    append_log(f'DONE seed={seed} {dataset_name} / {split_policy} / {model_name}: AP={metrics["average_precision"]:.4f}, top3={top}')
    return metric_row, importance

metric_frames = []
importance_frames = []
for seed in SEEDS:
    for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
        for split_policy in SPLIT_POLICIES:
            for model_name in MODEL_NAMES:
                try:
                    metrics, importance = run_one(seed, dataset_name, split_policy, model_name)
                    metric_frames.append(metrics)
                    importance_frames.append(importance)
                    write_csv_atomic(RESULTS_DIR / 'partial_metrics_latest.csv', pd.concat(metric_frames, ignore_index=True))
                    write_csv_atomic(RESULTS_DIR / 'partial_importance_latest.csv', pd.concat(importance_frames, ignore_index=True))
                except Exception as exc:
                    append_log(f'ERROR seed={seed} {dataset_name} / {split_policy} / {model_name}: {repr(exc)}')
                    write_json(CHECKPOINT_DIR / f'ERROR__seed{seed}__{dataset_name}__{split_policy}__{model_name}.json', {
                        'seed': seed,
                        'dataset': dataset_name,
                        'split_policy': split_policy,
                        'model': model_name,
                        'error': repr(exc),
                        'timestamp': now(),
                    })

metric_frames = [pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('seed*.csv'))]
importance_frames = [pd.read_csv(p) for p in sorted(IMPORTANCE_DIR.glob('importance__seed*.csv'))]
metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
importance = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'explanation_model_metrics.csv', metrics)
write_csv_atomic(RESULTS_DIR / 'feature_importance_rankings.csv', importance)
metrics

In [ ]:
importance = pd.read_csv(RESULTS_DIR / 'feature_importance_rankings.csv')

def vector_for(group):
    return group.set_index('raw_feature')['importance_normalized'].to_dict()

def topk_set(vec, k):
    ordered = sorted(vec.items(), key=lambda kv: (-kv[1], kv[0]))
    return set([name for name, _val in ordered[:min(k, len(ordered))]])

def compare_vectors(vec_a, vec_b, k):
    features = sorted(set(vec_a) | set(vec_b))
    arr_a = np.array([vec_a.get(f, 0.0) for f in features])
    arr_b = np.array([vec_b.get(f, 0.0) for f in features])
    top_a = topk_set(vec_a, k)
    top_b = topk_set(vec_b, k)
    union = top_a | top_b
    jaccard = len(top_a & top_b) / len(union) if union else np.nan
    weighted_overlap = float(np.minimum(arr_a, arr_b).sum())
    if len(features) > 1 and np.std(arr_a) > 0 and np.std(arr_b) > 0:
        rank_corr = float(spearmanr(arr_a, arr_b).correlation)
    else:
        rank_corr = np.nan
    return {
        'top_k': int(k),
        'topk_jaccard': float(jaccard),
        'explanation_instability_index': float(1.0 - jaccard) if not pd.isna(jaccard) else np.nan,
        'weighted_importance_overlap': weighted_overlap,
        'importance_spearman': rank_corr,
    }

seed_rows = []
protocol_rows = []

for (dataset, model, split_policy), part in importance.groupby(['dataset', 'model', 'split_policy']):
    by_seed = {seed: vector_for(g) for seed, g in part.groupby('seed')}
    for seed_a, seed_b in itertools.combinations(sorted(by_seed), 2):
        for k in TOP_K_VALUES:
            row = compare_vectors(by_seed[seed_a], by_seed[seed_b], k)
            row.update({
                'comparison_type': 'seed_pair',
                'dataset': dataset,
                'model': model,
                'split_policy': split_policy,
                'seed_a': seed_a,
                'seed_b': seed_b,
            })
            seed_rows.append(row)

for (dataset, model, seed), part in importance.groupby(['dataset', 'model', 'seed']):
    split_vectors = {sp: vector_for(g) for sp, g in part.groupby('split_policy')}
    if 'RANDOM_STRATIFIED_SAFE' in split_vectors and 'CHRONOLOGICAL_SAFE' in split_vectors:
        for k in TOP_K_VALUES:
            row = compare_vectors(split_vectors['RANDOM_STRATIFIED_SAFE'], split_vectors['CHRONOLOGICAL_SAFE'], k)
            row.update({
                'comparison_type': 'protocol_pair',
                'dataset': dataset,
                'model': model,
                'seed': seed,
                'split_policy_a': 'RANDOM_STRATIFIED_SAFE',
                'split_policy_b': 'CHRONOLOGICAL_SAFE',
            })
            protocol_rows.append(row)

seed_stability = pd.DataFrame(seed_rows)
protocol_stability = pd.DataFrame(protocol_rows)
write_csv_atomic(RESULTS_DIR / 'seed_explanation_stability_index.csv', seed_stability)
write_csv_atomic(RESULTS_DIR / 'protocol_explanation_stability_index.csv', protocol_stability)

seed_summary = (
    seed_stability
    .groupby(['dataset', 'model', 'split_policy', 'top_k'])
    .agg(
        comparisons=('topk_jaccard', 'size'),
        topk_jaccard_mean=('topk_jaccard', 'mean'),
        topk_jaccard_std=('topk_jaccard', 'std'),
        instability_mean=('explanation_instability_index', 'mean'),
        weighted_overlap_mean=('weighted_importance_overlap', 'mean'),
        spearman_mean=('importance_spearman', 'mean'),
    )
    .reset_index()
)

protocol_summary = (
    protocol_stability
    .groupby(['dataset', 'model', 'top_k'])
    .agg(
        comparisons=('topk_jaccard', 'size'),
        topk_jaccard_mean=('topk_jaccard', 'mean'),
        topk_jaccard_std=('topk_jaccard', 'std'),
        instability_mean=('explanation_instability_index', 'mean'),
        weighted_overlap_mean=('weighted_importance_overlap', 'mean'),
        spearman_mean=('importance_spearman', 'mean'),
    )
    .reset_index()
)

top_features = (
    importance
    .groupby(['dataset', 'model', 'split_policy', 'raw_feature'])
    .agg(
        mean_importance=('importance_normalized', 'mean'),
        std_importance=('importance_normalized', 'std'),
        mean_rank=('rank', 'mean'),
        top3_frequency=('rank', lambda s: float(np.mean(s <= 3))),
        top5_frequency=('rank', lambda s: float(np.mean(s <= 5))),
    )
    .reset_index()
    .sort_values(['dataset', 'model', 'split_policy', 'mean_importance'], ascending=[True, True, True, False])
)

write_csv_atomic(RESULTS_DIR / 'seed_explanation_stability_summary.csv', seed_summary)
write_csv_atomic(RESULTS_DIR / 'protocol_explanation_stability_summary.csv', protocol_summary)
write_csv_atomic(RESULTS_DIR / 'top_feature_stability_summary.csv', top_features)
protocol_summary

In [ ]:
seed_summary = pd.read_csv(RESULTS_DIR / 'seed_explanation_stability_summary.csv')
protocol_summary = pd.read_csv(RESULTS_DIR / 'protocol_explanation_stability_summary.csv')
top_features = pd.read_csv(RESULTS_DIR / 'top_feature_stability_summary.csv')

dataset_labels = {'D1_creditcard': 'D1', 'D2_fraudTest': 'D2', 'D3_PaySim': 'D3'}
model_labels = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM'}
split_labels = {'RANDOM_STRATIFIED_SAFE': 'Random split', 'CHRONOLOGICAL_SAFE': 'Chronological split'}

for df in [seed_summary, protocol_summary, top_features]:
    df['dataset_label'] = df['dataset'].map(dataset_labels)
    df['model_label'] = df['model'].map(model_labels)
if 'split_policy' in seed_summary.columns:
    seed_summary['split_label'] = seed_summary['split_policy'].map(split_labels)
if 'split_policy' in top_features.columns:
    top_features['split_label'] = top_features['split_policy'].map(split_labels)

plot_protocol = protocol_summary[protocol_summary['top_k'] == 5].copy()
plot_protocol['row_label'] = plot_protocol['dataset_label'] + ' - ' + plot_protocol['model_label']
heat = plot_protocol.pivot_table(index='row_label', values='instability_mean', aggfunc='mean')

fig, ax = plt.subplots(figsize=(4.8, max(3.0, 0.45 * len(heat))))
sns.heatmap(
    heat,
    annot=True,
    fmt='.3f',
    cmap='mako_r',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': '1 - Top-5 Jaccard'},
    annot_kws={'fontsize': 8},
    ax=ax,
)
ax.set_title('Protocol Explanation Instability')
ax.set_xlabel('')
ax.set_ylabel('Dataset and model')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
png = FIGURES_DIR / 'protocol_explanation_instability_heatmap.png'
pdf = FIGURES_DIR / 'protocol_explanation_instability_heatmap.pdf'
fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show()
append_log(f'Saved figure: {png}')

plot_seed = seed_summary[seed_summary['top_k'] == 5].copy()
fig, ax = plt.subplots(figsize=(7.4, 4.2))
sns.barplot(
    data=plot_seed,
    x='dataset_label',
    y='topk_jaccard_mean',
    hue='model_label',
    errorbar=None,
    edgecolor='black',
    linewidth=0.45,
    ax=ax,
)
ax.set_ylim(0, 1.02)
ax.set_xlabel('Dataset')
ax.set_ylabel('Mean Top-5 Jaccard across seed pairs')
ax.set_title('Seed-Level Explanation Stability')
ax.legend(title='Model', loc='best', frameon=True)
ax.grid(axis='y', linestyle='--', linewidth=0.45, alpha=0.55)
ax.grid(axis='x', visible=False)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
png = FIGURES_DIR / 'seed_explanation_stability_top5.png'
pdf = FIGURES_DIR / 'seed_explanation_stability_top5.pdf'
fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show()
append_log(f'Saved figure: {png}')

for dataset_label in ['D1', 'D2', 'D3']:
    part = top_features[(top_features['dataset_label'] == dataset_label) & (top_features['split_policy'] == 'RANDOM_STRATIFIED_SAFE')].copy()
    part = (
        part.sort_values(['model_label', 'mean_importance'], ascending=[True, False])
        .groupby('model_label')
        .head(8)
        .reset_index(drop=True)
    )
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    sns.barplot(
        data=part,
        y='raw_feature',
        x='mean_importance',
        hue='model_label',
        edgecolor='black',
        linewidth=0.45,
        ax=ax,
    )
    ax.set_xlabel('Mean normalized importance')
    ax.set_ylabel('Feature')
    ax.set_title(f'Top Strict Features: {dataset_label}')
    ax.legend(title='Model', loc='best', frameon=True)
    ax.grid(axis='x', linestyle='--', linewidth=0.45, alpha=0.55)
    ax.grid(axis='y', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    png = FIGURES_DIR / f'top_strict_features_{dataset_label}.png'
    pdf = FIGURES_DIR / f'top_strict_features_{dataset_label}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / 'explanation_model_metrics.csv')
importance = pd.read_csv(RESULTS_DIR / 'feature_importance_rankings.csv')
seed_stability = pd.read_csv(RESULTS_DIR / 'seed_explanation_stability_index.csv')
protocol_stability = pd.read_csv(RESULTS_DIR / 'protocol_explanation_stability_index.csv')
seed_summary = pd.read_csv(RESULTS_DIR / 'seed_explanation_stability_summary.csv')
protocol_summary = pd.read_csv(RESULTS_DIR / 'protocol_explanation_stability_summary.csv')
top_features = pd.read_csv(RESULTS_DIR / 'top_feature_stability_summary.csv')

run_summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'seeds': SEEDS,
    'models': MODEL_NAMES,
    'split_policies': SPLIT_POLICIES,
    'top_k_values': TOP_K_VALUES,
    'caps': CAPS,
    'metrics_file': str(RESULTS_DIR / 'explanation_model_metrics.csv'),
    'importance_file': str(RESULTS_DIR / 'feature_importance_rankings.csv'),
    'seed_index_file': str(RESULTS_DIR / 'seed_explanation_stability_index.csv'),
    'protocol_index_file': str(RESULTS_DIR / 'protocol_explanation_stability_index.csv'),
    'seed_summary_file': str(RESULTS_DIR / 'seed_explanation_stability_summary.csv'),
    'protocol_summary_file': str(RESULTS_DIR / 'protocol_explanation_stability_summary.csv'),
    'top_features_file': str(RESULTS_DIR / 'top_feature_stability_summary.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'importance_checkpoint_dir': str(IMPORTANCE_DIR),
    'n_metric_rows': int(len(metrics)),
    'n_importance_rows': int(len(importance)),
    'n_seed_stability_rows': int(len(seed_stability)),
    'n_protocol_stability_rows': int(len(protocol_stability)),
    'n_seed_summary_rows': int(len(seed_summary)),
    'n_protocol_summary_rows': int(len(protocol_summary)),
    'n_top_feature_rows': int(len(top_features)),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', run_summary)
run_summary

## Expected Runtime on Colab T4

Expected runtime is approximately **50-90 minutes**.

Why:

- Total model fits: 3 datasets x 2 split policies x 2 models x 5 seeds = 60 fits.
- Explanation extraction uses model-native tree importances, so it is lightweight after training.
- D3 PaySim is capped to 1,000,000 timeline-sampled rows by default.

If Google Drive is slow, allow up to **2 hours**. If Colab disconnects, rerun from the top. Existing metric and importance checkpoints are skipped.